In [1]:
import pandas as pd
import numpy as np



In [2]:
df = pd.read_csv("/content/01 covid_toy.csv")
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [3]:
print(df.cough.value_counts())
print()
print()
print(df.city.value_counts())

cough
Mild      62
Strong    38
Name: count, dtype: int64


city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64


In [4]:
df.isnull().sum()


,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [5]:
x = df.iloc[:,0:5]
y = df.drop(x,axis = 'columns')

print(x.head())

print()
print()
print()

print(y.head())

   age  gender  fever cough     city
0   60    Male  103.0  Mild  Kolkata
1   27    Male  100.0  Mild    Delhi
2   42    Male  101.0  Mild    Delhi
3   31  Female   98.0  Mild  Kolkata
4   65  Female  101.0  Mild   Mumbai



  has_covid
0        No
1       Yes
2        No
3        No
4        No


In [6]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2)
print(x_train.head())
print()
print()
print(y_train.head())

    age  gender  fever   cough       city
88    5  Female  100.0    Mild    Kolkata
36   38  Female  101.0    Mild  Bangalore
8    19  Female  100.0  Strong  Bangalore
50   19    Male  101.0    Mild      Delhi
2    42    Male  101.0    Mild      Delhi


   has_covid
88        No
36        No
8         No
50       Yes
2         No


In [7]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

In [8]:
si = SimpleImputer()
le = LabelEncoder()

In [9]:
#applying label encoding on class column

y_train.has_covid = le.fit_transform(y_train.has_covid)
y_test.has_covid = le.fit_transform(y_test['has_covid'])

In [12]:
x_train['fever'] = si.fit_transform(x_train[['fever']])


In [14]:
from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder()
x_train_cough = oe.fit_transform(x_train[['cough']])
x_test_cough = oe.fit_transform(x_test[['cough']])

In [15]:
from sklearn.preprocessing import OneHotEncoder

In [18]:
ohe = OneHotEncoder(drop = 'first',sparse_output = False)



In [20]:
x_train_gender_city = ohe.fit_transform(x_train[['gender','city']])
x_test_gender_city = ohe.transform(x_test[['gender','city']])

print(x_train_gender_city.shape)
print(x_test_gender_city.shape)

(80, 4)
(20, 4)


In [21]:
x_train

,age,gender,fever,cough,city
88,5,Female,100.0,Mild,Kolkata
36,38,Female,101.0,Mild,Bangalore
8,19,Female,100.0,Strong,Bangalore
50,19,Male,101.0,Mild,Delhi
2,42,Male,101.0,Mild,Delhi
...,...,...,...,...,...
64,42,Male,104.0,Mild,Mumbai
81,65,Male,99.0,Mild,Delhi
89,46,Male,103.0,Strong,Bangalore
4,65,Female,101.0,Mild,Mumbai


In [22]:
transformer = [('trans1',SimpleImputer(),['fever']),
               ('trans2',OrdinalEncoder(categories = [['Mild','Strong']]),['cough']),
               ('trans3',OneHotEncoder(sparse_output = False,drop = 'first'),['gender','city'])]

In [25]:
from sklearn.compose import ColumnTransformer

ct = ColumnTransformer(transformers = transformer,remainder = 'passthrough').set_output(transform = 'pandas')

In [26]:
x_train_trans = ct.fit_transform(x_train)


In [27]:
x_train_trans.sample(5)

,trans1__fever,trans2__cough,trans3__gender_Male,trans3__city_Delhi,trans3__city_Kolkata,trans3__city_Mumbai,remainder__age
3,98.000000,0.0,0.0,0.0,1.0,0.0,31
13,102.000000,0.0,1.0,0.0,0.0,0.0,64
54,99.000000,0.0,0.0,0.0,0.0,1.0,60
56,100.819444,1.0,1.0,0.0,1.0,0.0,71
81,99.000000,0.0,1.0,1.0,0.0,0.0,65


In [30]:
x_train_trans.shape

# We can see that wr're getting the same result as we got in above tedious method


(80, 7)

In [31]:
x_test_trans = ct.transform(x_test)


In [32]:
print(x_test_trans.sample(5))
print()
print()

print(x_test_trans.shape)

    trans1__fever  trans2__cough  trans3__gender_Male  trans3__city_Delhi  \
98           98.0            1.0                  0.0                 0.0   
61           98.0            1.0                  0.0                 0.0   
62          104.0            1.0                  0.0                 0.0   
31          103.0            0.0                  1.0                 0.0   
34          102.0            0.0                  1.0                 0.0   

    trans3__city_Kolkata  trans3__city_Mumbai  remainder__age  
98                   0.0                  1.0               5  
61                   0.0                  1.0              81  
62                   0.0                  0.0              56  
31                   1.0                  0.0              83  
34                   0.0                  1.0              74  


(20, 7)
